In [1]:
// load the formatting setup, the visualization functions, and the style sheet, install Z3 if necessary
#load "../setup.fsx"
#load "../visuals.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

# Sudoku with Boolean Variables

The Finnish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is defined below.  Empty fields are represented by the digit `0`.

In [3]:
let puzzle : int[][] =
    [| [| 8; 0; 0; 0; 0; 0; 0; 0; 0 |]
       [| 0; 0; 3; 6; 0; 0; 0; 0; 0 |]
       [| 0; 7; 0; 0; 9; 0; 2; 0; 0 |]
       [| 0; 5; 0; 0; 0; 7; 0; 0; 0 |]
       [| 0; 0; 0; 0; 4; 5; 7; 0; 0 |]
       [| 0; 0; 0; 1; 0; 0; 0; 3; 0 |]
       [| 0; 0; 1; 0; 0; 0; 0; 6; 8 |]
       [| 0; 0; 8; 5; 0; 0; 0; 1; 0 |]
       [| 0; 9; 0; 0; 0; 0; 4; 0; 0 |] |]

We will solve this Sudoku with `Z3` using Boolean variables, i.e. we encode the Sudoku in propositional logic as we did in the notebook `10-Sudoku.ipynb` of chapter 4.  We use the following variables:
* `Vrcd` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `var` takes three arguments `row`, `col`, and `digit` and returns a string that is interpreted as a variable name.

In [4]:
let var (row: int) (col: int) (digit: int) : string =
    sprintf "V%d%d%d" row col digit

In [5]:
var 1 2 3

"V123"

The function `createVarDict` takes no arguments.  It returns a map that maps every triple `(row, col, digit)` to the corresponding Boolean `Z3` variable.

In [6]:
let createVarDict () : Map<int * int * int, BoolExpr> =
    Map [ for row in 1 .. 9 do
            for col in 1 .. 9 do
                for digit in 1 .. 9 ->
                    ((row, col, digit), Bool (var row col digit)) ]

In [7]:
let VarDict = createVarDict ()
VarDict.Count

729

The function `atMostOne` takes one argument:
- `S` is a list of Boolean `Z3` variables.

It returns a list of formulas expressing the fact that at most one of the variables of `S` is true.  The pairs of variables are compared via their names, so that every pair is only considered once.

In [8]:
let atMostOne (S: BoolExpr list) : BoolExpr list =
    [ for p in S do
        for q in S do
            if string p < string q then
                Not (And [ p; q ]) ]

The function `atLeastOne` takes a list `S` of Boolean `Z3` variables.  It returns a list containing a formula expressing the fact that at least one of the variables of `S` is true.

In [9]:
let atLeastOne (S: BoolExpr list) : BoolExpr list =
    [ Or S ]

The function `exactlyOne` takes a list `S` of Boolean `Z3` variables.  It returns a list of formulas expressing the fact that exactly one of the variables of `S` is true.

In [10]:
let exactlyOne (S: BoolExpr list) : BoolExpr list =
    atMostOne S @ atLeastOne S

In [11]:
let a, b, c = Bool "a", Bool "b", Bool "c"
exactlyOne [ a; b; c ]

[¬(a ∧ b); ¬(a ∧ c); ¬(b ∧ c); a ∨ b ∨ c]

The function `allDifferent` takes two arguments:
- `L` is a list of pairs of the form `(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$,
- `VarDict` is the map from triples to variables.

It returns a list of formulas expressing that all Sudoku fields specified by the coordinate pairs in `L` take different digits as values.

In [12]:
let allDifferent (L: (int * int) list) (VarDict: Map<int * int * int, BoolExpr>) : BoolExpr list =
    List.concat [ for digit in 1 .. 9 -> exactlyOne [ for (row, col) in L -> VarDict[(row, col, digit)] ] ]

In [13]:
(allDifferent [ for c in 1 .. 9 -> (1, c) ] VarDict).Length

333

The function `exactlyOneDigit` takes three arguments `row`, `col`, and `VarDict`.  The first two arguments specify a field of the Sudoku.  The function returns a list of formulas specifying that exactly one of the variables

* `V{row}{col}1`, `V{row}{col}2`, $\cdots$, `V{row}{col}9`

is `true`.

In [14]:
let exactlyOneDigit (row: int) (col: int) (VarDict: Map<int * int * int, BoolExpr>) : BoolExpr list =
    exactlyOne [ for digit in 1 .. 9 -> VarDict[(row, col, digit)] ]

In [15]:
exactlyOneDigit 1 1 VarDict

[¬(V111 ∧ V112); ¬(V111 ∧ V113); ¬(V111 ∧ V114); ¬(V111 ∧ V115); ¬(V111 ∧ V116); ¬(V111 ∧ V117); ¬(V111 ∧ V118); ¬(V111 ∧ V119); ¬(V112 ∧ V113); ¬(V112 ∧ V114); ¬(V112 ∧ V115); ¬(V112 ∧ V116); ¬(V112 ∧ V117); ¬(V112 ∧ V118); ¬(V112 ∧ V119); ¬(V113 ∧ V114); ¬(V113 ∧ V115); ¬(V113 ∧ V116); ¬(V113 ∧ V117); ¬(V113 ∧ V118); ¬(V113 ∧ V119); ¬(V114 ∧ V115); ¬(V114 ∧ V116); ¬(V114 ∧ V117); ¬(V114 ∧ V118); ¬(V114 ∧ V119); ¬(V115 ∧ V116); ¬(V115 ∧ V117); ¬(V115 ∧ V118); ¬(V115 ∧ V119); ¬(V116 ∧ V117); ¬(V116 ∧ V118); ¬(V116 ∧ V119); ¬(V117 ∧ V118); ¬(V117 ∧ V119); ¬(V118 ∧ V119); V111 ∨ V112 ∨ V113 ∨ V114 ∨ V115 ∨ V116 ∨ V117 ∨ V118 ∨ V119]

The function `constraintsFromPuzzle` takes the map `VarDict`.  It returns a list containing a formula stating that the variables corresponding to numbers that are already given in the Sudoku puzzle are true.

In [16]:
let constraintsFromPuzzle (VarDict: Map<int * int * int, BoolExpr>) : BoolExpr list =
    [ And [ for row in 0 .. 8 do
              for col in 0 .. 8 do
                  if puzzle[row][col] <> 0 then
                      VarDict[(row + 1, col + 1, puzzle[row][col])] ] ]

In [17]:
constraintsFromPuzzle VarDict

[V118 ∧ V233 ∧ V246 ∧ V327 ∧ V359 ∧ V372 ∧ V425 ∧ V467 ∧ V554 ∧ V565 ∧ V577 ∧ V641 ∧ V683 ∧ V731 ∧ V786 ∧ V798 ∧ V838 ∧ V845 ∧ V881 ∧ V929 ∧ V974]

The function `allConstraints` takes the map `VarDict`.  It returns a list of formulas that encodes the given Sudoku.

In [18]:
let allConstraints (VarDict: Map<int * int * int, BoolExpr>) : BoolExpr list =
    let L = [ 1 .. 9 ]
    // the constraints from the puzzle have to be satisfied
    let mutable Clauses = constraintsFromPuzzle VarDict
    // there is exactly one digit in every field
    for row in L do
        for col in L do
            Clauses <- Clauses @ exactlyOneDigit row col VarDict
    // all entries in a row are unique
    for row in L do
        Clauses <- Clauses @ allDifferent [ for col in L -> (row, col) ] VarDict
    // all entries in a column are unique
    for col in L do
        Clauses <- Clauses @ allDifferent [ for row in L -> (row, col) ] VarDict
    // all entries in a 3x3 square are unique
    for r in 0 .. 2 do
        for c in 0 .. 2 do
            Clauses <- Clauses @ allDifferent [ for row in 1 .. 3 do
                                                  for col in 1 .. 3 -> (r * 3 + row, c * 3 + col) ] VarDict
    Clauses

In [19]:
(allConstraints VarDict).Length

11989

The function `sudoku` takes no arguments.  It solves the Sudoku and returns the solution as a map assigning the digits to the positions `(row, col)`.  If the Sudoku can not be solved, a message is printed and the empty map is returned.  The function `boolValue` is defined in `z3.fsx`.  It returns the value of a Boolean variable in a model.

In [20]:
let sudoku () : Map<int * int, int> =
    let VarDict = createVarDict ()
    let S = newSolver ()
    S.Add(Array.ofList (allConstraints VarDict))
    match S.Check() with
    | Status.SATISFIABLE ->
        let M = S.Model
        Map [ for KeyValue((row, col, digit), v) in VarDict do
                if boolValue M v then ((row, col), digit) ]
    | Status.UNSATISFIABLE ->
        printfn "The problem is not solvable."
        Map.empty
    | _ ->
        printfn "Z3 cannot determine whether the problem is solvable."
        Map.empty

In [21]:
#!time
let Solution = sudoku ()

Wall time: 206.6086ms

In [22]:
Solution

map [((1, 1), 8); ((1, 2), 1); ((1, 3), 2); ((1, 4), 7); ((1, 5), 5); ((1, 6), 3); ((1, 7), 6); ((1, 8), 4); ((1, 9), 9); ((2, 1), 9); ((2, 2), 4); ((2, 3), 3); ((2, 4), 6); ((2, 5), 8); ((2, 6), 2); ((2, 7), 1); ((2, 8), 7); ((2, 9), 5); ((3, 1), 6); ((3, 2), 7); ((3, 3), 5); ((3, 4), 4); ((3, 5), 9); ((3, 6), 1); ((3, 7), 2); ((3, 8), 8); ((3, 9), 3); ((4, 1), 1); ((4, 2), 5); ((4, 3), 4); ((4, 4), 2); ((4, 5), 3); ((4, 6), 7); ((4, 7), 8); ((4, 8), 9); ((4, 9), 6); ((5, 1), 3); ((5, 2), 6); ((5, 3), 9); ((5, 4), 8); ((5, 5), 4); ((5, 6), 5); ((5, 7), 7); ((5, 8), 2); ((5, 9), 1); ((6, 1), 2); ((6, 2), 8); ((6, 3), 7); ((6, 4), 1); ((6, 5), 6); ((6, 6), 9); ((6, 7), 5); ((6, 8), 3); ((6, 9), 4); ((7, 1), 5); ((7, 2), 2); ((7, 3), 1); ((7, 4), 9); ((7, 5), 7); ((7, 6), 4); ((7, 7), 3); ((7, 8), 6); ((7, 9), 8); ((8, 1), 4); ((8, 2), 3); ((8, 3), 8); ((8, 4), 5); ((8, 5), 2); ((8, 6), 6); ((8, 7), 9); ((8, 8), 1); ((8, 9), 7); ((9, 1), 7); ((9, 2), 9); ((9, 3), 6); ((9, 4), 3); ((9, 5)

## Graphical Representation

The function `showSolution` takes two arguments:
- `Solution` is a map assigning the digits to the positions `(row, col)` of the Sudoku,
- `width` is the width of the grid as a CSS length.

It displays the Sudoku.  The digits that were given in the puzzle are shown in black, the digits computed by the solver are shown in blue.  The function `sudokuGrid` is defined in the file `visuals.fsx`.

In [23]:
let showSolution (Solution: Map<int * int, int>) (width: string) =
    sudokuGrid puzzle Solution width

In [24]:
showSolution Solution "50%"

8,1,2,7,5,3,6,4,9
9,4,3,6,8,2,1,7,5
6,7,5,4,9,1,2,8,3
1,5,4,2,3,7,8,9,6
3,6,9,8,4,5,7,2,1
2,8,7,1,6,9,5,3,4
5,2,1,9,7,4,3,6,8
4,3,8,5,2,6,9,1,7
7,9,6,3,1,8,4,5,2
